# Tutorial 8: Using Prefab Pipelines

Building a pipeline from scratch gives you full control, but PhenoTypic
also ships **prefab pipelines** — pre-configured `ImagePipeline` subclasses
tuned for common organisms and plate types. In this tutorial you will
survey the available prefabs, apply one, and compare results.

**What you will learn:**

1. What prefab pipelines are and when to use them
2. Survey the available prefabs
3. Apply a prefab pipeline
4. Compare results from different prefabs

## Imports

In [1]:
from phenotypic.data import load_synth_yeast_plate

## Available Prefab Pipelines

Each prefab is an `ImagePipeline` subclass with operations and measurements
already configured. Choose based on your organism and plate conditions.

| Prefab | Best for | Strategy |
|--------|----------|----------|
| `HeavyOtsuPipeline` | General-purpose yeast, clean plates | Multi-stage Otsu with refinement |
| `HeavyWatershedPipeline` | Touching or clustered colonies | Watershed segmentation |
| `RoundPeaksPipeline` | Round colonies, lightweight | Peak detection |
| `HeavyRoundPeaksPipeline` | Round colonies needing refinement | Extended peak detection |
| `FilamentousFungiPipeline` | Filamentous fungi (*Neurospora*, etc.) | BM3D denoise + specialized detector |
| `GridSectionPipeline` | Pre-tiled grid sections | Section-level processing |

## Apply HeavyOtsuPipeline

Let's start with the most general-purpose option — `HeavyOtsuPipeline`.
It chains Gaussian blur, CLAHE, median filtering, Sobel edge enhancement,
Otsu detection, and several refinement steps (morphological opening,
border removal, small-object removal, mask fill).

In [2]:
from phenotypic.prefab import HeavyOtsuPipeline

plate = load_synth_yeast_plate()
heavy_otsu = HeavyOtsuPipeline()
result_otsu = heavy_otsu.apply(plate)
result_otsu.dash(overlay=True)

PNG file does not have exif data.


In [3]:
print(f"HeavyOtsuPipeline detected {result_otsu.num_objects} colonies")

HeavyOtsuPipeline detected 96 colonies


## Apply RoundPeaksPipeline

Now let's try `RoundPeaksPipeline` — a lighter approach that uses peak
detection to find circular colonies. It is faster but may miss irregular
or faint colonies.

In [4]:
from phenotypic.prefab import RoundPeaksPipeline

plate2 = load_synth_yeast_plate()
round_peaks = RoundPeaksPipeline()
result_rp = round_peaks.apply(plate2)
result_rp.dash(overlay=True)

PNG file does not have exif data.


In [5]:
print(f"RoundPeaksPipeline detected {result_rp.num_objects} colonies")

RoundPeaksPipeline detected 96 colonies


## Compare

Different prefabs produce different results on the same plate. The best
choice depends on your colonies, your imaging conditions, and what you
need to measure. A quick visual comparison and colony count helps you
decide.

In [6]:
print(f"HeavyOtsuPipeline:   {result_otsu.num_objects} colonies")
print(f"RoundPeaksPipeline:  {result_rp.num_objects} colonies")

HeavyOtsuPipeline:   96 colonies
RoundPeaksPipeline:  96 colonies


## Prefabs Include Measurements

Prefab pipelines come with measurements pre-configured, so you can call
`.apply_and_measure()` directly — no need to add your own `meas` list.

In [7]:
plate3 = load_synth_yeast_plate()
df = heavy_otsu.apply_and_measure(plate3)
print(f"{len(df)} colonies measured across {df.shape[1]} features")
df.head()

PNG file does not have exif data.


96 colonies measured across 179 features


,Metadata_FileSuffix,Metadata_BitDepth,Metadata_ImageType,Metadata_ImageName,ObjectLabel,Bbox_CenterRR,Bbox_CenterCC,Bbox_MinRR,Bbox_MinCC,Bbox_MaxRR,...,Intensity_MaximumIntensity,Intensity_MeanIntensity,Intensity_MedianIntensity,Intensity_StandardDeviationIntensity,Intensity_CoefficientVarianceIntensity,Intensity_LowerQuartileIntensity,Intensity_UpperQuartileIntensity,Intensity_InterquartileRangeIntensity,Intensity_Density,Intensity_ConvexDensity
0,.png,8,GridImage,Synthetic96PlateWithObjects,1,66.379006,675.189503,38,652,96,...,0.943073,0.775636,0.911701,0.169254,0.218314,0.557704,0.923465,0.365762,0.775636,10.005758
1,.png,8,GridImage,Synthetic96PlateWithObjects,2,67.258437,60.400832,39,38,96,...,0.927387,0.777088,0.899936,0.161844,0.208367,0.561625,0.911701,0.350075,0.777088,9.988400
2,.png,8,GridImage,Synthetic96PlateWithObjects,3,67.774661,122.493213,39,99,98,...,0.931309,0.768554,0.903858,0.169159,0.220201,0.552949,0.915622,0.362673,0.768554,10.029955
3,.png,8,GridImage,Synthetic96PlateWithObjects,4,66.741818,615.873247,39,594,94,...,0.946995,0.776710,0.911701,0.173114,0.222997,0.560792,0.931309,0.370516,0.776710,9.446719
4,.png,8,GridImage,Synthetic96PlateWithObjects,5,67.689404,737.900864,39,714,98,...,0.939985,0.781367,0.914506,0.165666,0.212117,0.561625,0.919544,0.357918,0.781367,10.150514


## Summary

Prefab pipelines are the fastest path from plate image to results:

- **`HeavyOtsuPipeline`** — robust general-purpose detection with refinement
- **`RoundPeaksPipeline`** — lightweight peak-based detection for round colonies
- **`HeavyWatershedPipeline`** — for touching or clustered colonies
- **`FilamentousFungiPipeline`** — specialized for branching fungal morphology
- All prefabs support `.apply()`, `.apply_and_measure()`, `.to_json()`, etc.

Choose based on your organism, plate conditions, and desired accuracy.
When a prefab is close but not quite right, use it as a starting point
and customize the parameters.

**Next up:** [Tutorial 9: Diagnosing Image Quality](09_diagnosing_image_quality.ipynb) —
assess plate quality before choosing a pipeline.